In [ ]:
# ============================================================================
# CELL 1: Install Packages
# ============================================================================
!pip install torch transformers==4.47.1 accelerate==0.26.1 scikit-learn numpy pandas tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 121.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 270.9/270.9 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 52.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 117.3 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
  Attempting uninstall: accelerate
    Found existing installatio

In [ ]:
# ============================================================================
# CELL 2: Imports
# ============================================================================
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import random
import re
import warnings
warnings.filterwarnings('ignore')

def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

Using device: cuda
GPU: Tesla T4
GPU Memory: 14.56 GB


In [ ]:
# ============================================================================
# CELL 3: Load Data
# ============================================================================
train_path = "train_data (1).json"
test_path = "test_data_subtask_1 (1).json"

with open(train_path, 'r') as f:
    train_data = json.load(f)

with open(test_path, 'r') as f:
    test_data = json.load(f)

train_df = pd.DataFrame(train_data)
test_df = pd.DataFrame(test_data)

print(f"Training samples: {len(train_df)}")
print(f"Test samples: {len(test_df)}")
print(f"\nValidity distribution:\n{train_df['validity'].value_counts()}")
print(f"\nPlausibility distribution:\n{train_df['plausibility'].value_counts()}")

# Analyze 4 conditions
conditions = train_df.groupby(['plausibility', 'validity']).size()
print(f"\n4-Condition Distribution (Original):")
print(conditions)

Training samples: 960
Test samples: 191

Validity distribution:
validity
False    480
True     480
Name: count, dtype: int64

Plausibility distribution:
plausibility
False    486
True     474
Name: count, dtype: int64

4-Condition Distribution (Original):
plausibility  validity
False         False       246
              True        240
True          False       234
              True        240
dtype: int64


In [ ]:
# ============================================================================
# CELL 4: PROPER Train/Val Split BEFORE Augmentation
# ============================================================================
print("\n" + "="*80)
print("Step 1: Split ORIGINAL Data First (Prevent Data Leakage)")
print("="*80)

# Create condition column for stratification
train_df['condition'] = (
    train_df['plausibility'].astype(str) + '_' +
    train_df['validity'].astype(str)
)

# Split ORIGINAL data first
train_indices, val_indices = train_test_split(
    range(len(train_df)),
    test_size=0.15,
    random_state=42,
    stratify=train_df['condition'].values
)

train_df_original = train_df.iloc[train_indices].copy()
val_df_original = train_df.iloc[val_indices].copy()

print(f"\nOriginal data split:")
print(f"  Training: {len(train_df_original)}")
print(f"  Validation: {len(val_df_original)}")
print(f"\n CRITICAL: Augmentation will ONLY be applied to training data")
print(f"   Validation stays ORIGINAL to test true generalization")


Step 1: Split ORIGINAL Data First (Prevent Data Leakage)

Original data split:
  Training: 816
  Validation: 144

 CRITICAL: Augmentation will ONLY be applied to training data
   Validation stays ORIGINAL to test true generalization


In [ ]:
# ============================================================================
# CELL 5: Template-Based Augmentation (TRAINING ONLY)
# ============================================================================
print("\n" + "="*80)
print("Step 2: Augment ONLY Training Data")
print("="*80)

class TemplateAugmenter:
    """Generate variations that preserve logical structure"""

    def __init__(self):
        self.entity_pools = {
            'vehicles': ['car', 'bicycle', 'motorcycle', 'truck', 'bus', 'train', 'airplane', 'boat'],
            'buildings': ['house', 'building', 'tower', 'castle', 'barn', 'shed', 'cabin', 'mansion'],
            'animals': ['dog', 'cat', 'horse', 'bird', 'fish', 'rabbit', 'lion', 'tiger'],
            'people': ['person', 'human', 'individual', 'citizen', 'student', 'teacher', 'worker', 'doctor'],
            'objects': ['book', 'table', 'chair', 'tool', 'device', 'machine', 'instrument', 'gadget'],
            'food': ['fruit', 'vegetable', 'meat', 'drink', 'meal', 'snack', 'dish', 'dessert'],
            'nature': ['tree', 'plant', 'flower', 'rock', 'mountain', 'river', 'ocean', 'forest'],
        }

    def identify_entities(self, text):
        """Extract main entities"""
        entities = []
        text_lower = text.lower()

        for category, entity_list in self.entity_pools.items():
            for entity in entity_list:
                if entity in text_lower:
                    entities.append((entity, category))

        return list(set(entities))[:2]  # Max 2 entities

    def replace_entities(self, text, old_entities, new_entities):
        """Case-preserving replacement"""
        result = text

        for (old, _), (new, _) in zip(old_entities, new_entities):
            result = re.sub(re.escape(old), new, result, flags=re.IGNORECASE)
            result = re.sub(re.escape(old.capitalize()), new.capitalize(), result)

        return result

    def generate_variations(self, text, validity, plausibility, n_variations=2):
        """Generate n variations with different entities"""
        variations = []
        entities = self.identify_entities(text)

        if not entities:
            return variations

        for i in range(n_variations):
            new_entities = []

            for old_entity, old_category in entities:
                # Pick new entity from same category
                available = [e for e in self.entity_pools[old_category] if e != old_entity]
                if available:
                    new_entity = random.choice(available)
                    new_entities.append((new_entity, old_category))
                else:
                    new_entities.append((old_entity, old_category))

            if len(new_entities) == len(entities):
                new_text = self.replace_entities(text, entities, new_entities)

                if new_text != text:
                    variations.append({
                        'syllogism': new_text,
                        'validity': validity,
                        'plausibility': plausibility
                    })

        return variations

# Augment ONLY training data
augmenter = TemplateAugmenter()
augmented_train = []
variation_count = 0

for idx, row in tqdm(train_df_original.iterrows(), total=len(train_df_original), desc="Augmenting training"):
    # Add original
    augmented_train.append(row.to_dict())

    # Generate 2 variations
    variations = augmenter.generate_variations(
        row['syllogism'],
        row['validity'],
        row['plausibility'],
        n_variations=2
    )

    augmented_train.extend(variations)
    variation_count += len(variations)

train_df_augmented = pd.DataFrame(augmented_train)
train_df_augmented = train_df_augmented.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\n✓ Training data augmentation:")
print(f"  Original training: {len(train_df_original)}")
print(f"  Generated variations: {variation_count}")
print(f"  Total training: {len(train_df_augmented)}")
print(f"  Expansion: {len(train_df_augmented)/len(train_df_original):.2f}x")

print(f"\n✓ Validation data (UNCHANGED):")
print(f"  Validation samples: {len(val_df_original)}")
print(f"  Status: NO AUGMENTATION (true generalization test)")


Step 2: Augment ONLY Training Data


Augmenting training: 100%|██████████| 816/816 [00:00<00:00, 9166.72it/s]


✓ Training data augmentation:
  Original training: 816
  Generated variations: 1224
  Total training: 2040
  Expansion: 2.50x

✓ Validation data (UNCHANGED):
  Validation samples: 144
  Status: NO AUGMENTATION (true generalization test)


In [ ]:
# ============================================================================
# CELL 6: Oversample Implausible in Training Only
# ============================================================================
print("\n" + "="*80)
print("Step 3: Oversample Implausible (Training Only)")
print("="*80)

# Separate by plausibility in augmented training data
plausible_train = train_df_augmented[train_df_augmented['plausibility'] == True]
implausible_train = train_df_augmented[train_df_augmented['plausibility'] == False]

print(f"\nBefore oversampling:")
print(f"  Plausible: {len(plausible_train)}")
print(f"  Implausible: {len(implausible_train)}")

# Oversample implausible 2x
implausible_oversampled = pd.concat([implausible_train, implausible_train], ignore_index=True)

# Combine
train_df_final = pd.concat([plausible_train, implausible_oversampled], ignore_index=True)
train_df_final = train_df_final.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\nAfter oversampling:")
print(f"  Total training: {len(train_df_final)}")

# Add dummy counterfactuals
train_df_final['counterfactual'] = train_df_final['syllogism']
val_df_original['counterfactual'] = val_df_original['syllogism']

# Final data
train_df = train_df_final
val_df = val_df_original

conditions_train = train_df.groupby(['plausibility', 'validity']).size()
conditions_val = val_df.groupby(['plausibility', 'validity']).size()

print(f"\nFinal 4-Condition Distribution:")
print(f"  TRAINING (augmented + oversampled):")
for cond, count in conditions_train.items():
    print(f"    {cond}: {count}")
print(f"  VALIDATION (original, no augmentation):")
for cond, count in conditions_val.items():
    print(f"    {cond}: {count}")

print(f"\n KEY: Validation = original data → Tests true generalization")


Step 3: Oversample Implausible (Training Only)

Before oversampling:
  Plausible: 1035
  Implausible: 1005

After oversampling:
  Total training: 3045

Final 4-Condition Distribution:
  TRAINING (augmented + oversampled):
    (False, False): 1054
    (False, True): 956
    (True, False): 529
    (True, True): 506
  VALIDATION (original, no augmentation):
    (False, False): 37
    (False, True): 36
    (True, False): 35
    (True, True): 36

 KEY: Validation = original data → Tests true generalization


In [ ]:
# ============================================================================
# CELL 7: Dataset Class
# ============================================================================
class SyllogismDataset(Dataset):
    def __init__(self, texts, counterfactuals, validity_labels, plausibility_labels, tokenizer, max_length=512):
        self.texts = texts
        self.counterfactuals = counterfactuals
        self.validity_labels = validity_labels
        self.plausibility_labels = plausibility_labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoded_orig = self.tokenizer(
            self.texts[idx],
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )

        encoded_cf = self.tokenizer(
            self.counterfactuals[idx],
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )

        return {
            'input_ids_orig': encoded_orig['input_ids'].squeeze(0),
            'attention_mask_orig': encoded_orig['attention_mask'].squeeze(0),
            'input_ids_cf': encoded_cf['input_ids'].squeeze(0),
            'attention_mask_cf': encoded_cf['attention_mask'].squeeze(0),
            'validity_label': torch.tensor(self.validity_labels[idx], dtype=torch.long),
            'plausibility_label': torch.tensor(self.plausibility_labels[idx], dtype=torch.long)
        }

In [ ]:
# ============================================================================
# CELL 8: Focal Loss
# ============================================================================
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=None, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.gamma = gamma
        self.alpha = alpha
        self.reduction = reduction

    def forward(self, inputs, targets):
     inputs = inputs.float()  # ensure float32
     ce_loss = F.cross_entropy(inputs, targets, reduction='none')
     ce_loss = torch.clamp(ce_loss, max=100.0)  # prevent exp overflow
     p_t = torch.exp(-ce_loss)
     focal_loss = (1 - p_t) ** self.gamma * ce_loss

     if self.alpha is not None:
        alpha_t = self.alpha[targets]
        focal_loss = alpha_t * focal_loss

     if self.reduction == 'mean':
        return focal_loss.mean()
     elif self.reduction == 'sum':
        return focal_loss.sum()
     else:
        return focal_loss

In [ ]:
# ============================================================================
# CELL 9: Gradient Reversal Layer
# ============================================================================
class GradientReversalFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambda_):
        ctx.lambda_ = lambda_
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return grad_output.neg() * ctx.lambda_, None

class GradientReversalLayer(nn.Module):
    def __init__(self, lambda_=1.0):
        super(GradientReversalLayer, self).__init__()
        self.lambda_ = lambda_

    def forward(self, x):
        return GradientReversalFunction.apply(x, self.lambda_)

In [ ]:
# ============================================================================
# CELL 10: MLA-CI Model
# ============================================================================
class MLACIModel(nn.Module):
    def __init__(self, model_name, lambda_adv=1.0, dropout=0.2):
        super(MLACIModel, self).__init__()
        self.encoder = AutoModel.from_pretrained(
              model_name,
              output_hidden_states=True,
             low_cpu_mem_usage=False  # Force eager loading, disables accelerate's lazy materialization
             )
        self.encoder = self.encoder.float()  # Cast AFTER loading, not during

        hidden_size = self.encoder.config.hidden_size
        self.combined_size = hidden_size * 3
        self.grl = GradientReversalLayer(lambda_=lambda_adv)

        self.validity_classifier = nn.Sequential(
            nn.Linear(self.combined_size, 768),
            nn.LayerNorm(768),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(768, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 2)
        )

        self.plausibility_adversary = nn.Sequential(
            nn.Linear(self.combined_size, 768),
            nn.LayerNorm(768),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(768, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 2)
        )

    def extract_multi_layer_features(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        hidden_states = outputs.hidden_states

        layer_2 = hidden_states[2][:, 0, :]
        layer_6 = hidden_states[6][:, 0, :]
        layer_neg2 = hidden_states[-2][:, 0, :]

        combined = torch.cat([layer_2, layer_6, layer_neg2], dim=1)
        return combined.float()

    def forward(self, input_ids, attention_mask, return_features=False):
        features = self.extract_multi_layer_features(input_ids, attention_mask)

        validity_logits = self.validity_classifier(features)

        reversed_features = self.grl(features)
        plausibility_logits = self.plausibility_adversary(reversed_features)

        if return_features:
            return validity_logits, plausibility_logits, features
        return validity_logits, plausibility_logits

In [ ]:
# ============================================================================
# CELL 11: Data Preparation (Using Pre-Split Data)
# ============================================================================
MODEL_NAME = "microsoft/deberta-v3-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Use the already-split data (NO SECOND SPLIT!)
# train_df = augmented + oversampled training data
# val_df = original validation data (no augmentation)

print(f"\n Using Pre-Split Data:")
print(f"  Training: {len(train_df)} samples (augmented + oversampled)")
print(f"  Validation: {len(val_df)} samples (original, no augmentation)")

# Prepare training data
train_texts = train_df['syllogism'].tolist()
train_cf = train_df['counterfactual'].tolist()
train_validity = train_df['validity'].astype(int).values
train_plausibility = train_df['plausibility'].astype(int).values

# Prepare validation data
val_texts = val_df['syllogism'].tolist()
val_cf = val_df['counterfactual'].tolist()
val_validity = val_df['validity'].astype(int).values
val_plausibility = val_df['plausibility'].astype(int).values

# Create datasets
train_dataset = SyllogismDataset(train_texts, train_cf, train_validity, train_plausibility, tokenizer)
val_dataset = SyllogismDataset(val_texts, val_cf, val_validity, val_plausibility, tokenizer)

# Create dataloaders
BATCH_SIZE = 4
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"\nDataLoaders created:")
print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches: {len(val_loader)}")
print(f"  Batch size: {BATCH_SIZE}")

# Initialize model
model = MLACIModel(MODEL_NAME, lambda_adv=1.0, dropout=0.2)
model = model.to(device)

print(f"  Model parameters: {sum(p.numel() for p in model.parameters()):,}")

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]


 Using Pre-Split Data:
  Training: 3045 samples (augmented + oversampled)
  Validation: 144 samples (original, no augmentation)

DataLoaders created:
  Train batches: 762
  Val batches: 36
  Batch size: 4


pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

  Model parameters: 187,769,860


In [ ]:
# ============================================================================
# CELL A: Imports & helpers (run after notebook cells 1-11)
# ============================================================================
import json
import copy
import gc
import numpy as np

print("✓ Ready. Using existing variables from cells 1-11:")
print(f"  train_df_augmented: {len(train_df_augmented)} samples (aug, no oversample)")
print(f"  val_df_original: {len(val_df_original)} samples")

✓ Ready. Using existing variables from cells 1-11:
  train_df_augmented: 2040 samples (aug, no oversample)
  val_df_original: 144 samples


In [ ]:
# ============================================================================
# CELL B: Prepare data — Augmented only, NO oversampling
# ============================================================================

# Use augmented training data WITHOUT oversampling
train_df_optimal = train_df_augmented.copy()  # 2040 samples

# Ensure counterfactual column exists
if 'counterfactual' not in train_df_optimal.columns:
    train_df_optimal['counterfactual'] = train_df_optimal['syllogism']

val_df_opt = val_df_original.copy()
if 'counterfactual' not in val_df_opt.columns:
    val_df_opt['counterfactual'] = val_df_opt['syllogism']

print(f"Training data: {len(train_df_optimal)} samples (augmented, NO oversampling)")
print(f"Validation data: {len(val_df_opt)} samples (original, untouched)")

# Show distribution
conds = train_df_optimal.groupby(['plausibility', 'validity']).size()
print(f"\nTraining distribution:")
for cond, count in conds.items():
    print(f"  {cond}: {count}")

Training data: 2040 samples (augmented, NO oversampling)
Validation data: 144 samples (original, untouched)

Training distribution:
  (False, False): 527
  (False, True): 478
  (True, False): 529
  (True, True): 506


In [ ]:
# ============================================================================
# CELL C: Create datasets and dataloaders
# ============================================================================

# Training data
opt_train_texts = train_df_optimal['syllogism'].tolist()
opt_train_cf = train_df_optimal['counterfactual'].tolist()
opt_train_validity = train_df_optimal['validity'].astype(int).values
opt_train_plausibility = train_df_optimal['plausibility'].astype(int).values

# Validation data
opt_val_texts = val_df_opt['syllogism'].tolist()
opt_val_cf = val_df_opt['counterfactual'].tolist()
opt_val_validity = val_df_opt['validity'].astype(int).values
opt_val_plausibility = val_df_opt['plausibility'].astype(int).values

# Datasets
opt_train_dataset = SyllogismDataset(
    opt_train_texts, opt_train_cf, opt_train_validity,
    opt_train_plausibility, tokenizer
)
opt_val_dataset = SyllogismDataset(
    opt_val_texts, opt_val_cf, opt_val_validity,
    opt_val_plausibility, tokenizer
)

# Dataloaders
BATCH_SIZE = 4
opt_train_loader = DataLoader(opt_train_dataset, batch_size=BATCH_SIZE, shuffle=True)
opt_val_loader = DataLoader(opt_val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train batches: {len(opt_train_loader)}")
print(f"Val batches: {len(opt_val_loader)}")

Train batches: 510
Val batches: 36


In [ ]:
# ============================================================================
# CELL D: Initialize model — Multi-layer YES, Adversarial NO (λ=0)
# ============================================================================

# Uses MLACIModel (multi-layer) but with lambda_adv=0 (no adversarial)
opt_model = MLACIModel(MODEL_NAME, lambda_adv=0.0, dropout=0.2)
opt_model = opt_model.to(device)

# Standard CrossEntropy — NO focal loss
opt_criterion = nn.CrossEntropyLoss()
adv_criterion = nn.CrossEntropyLoss()  # Still needed for forward pass, but λ=0 means no gradient

# Optimizer
LAMBDA_ADV = 0.0
LR = 2e-5
EPOCHS = 4
ACCUMULATION_STEPS = 4

opt_optimizer = torch.optim.AdamW(opt_model.parameters(), lr=LR, weight_decay=0.01)
total_steps = (len(opt_train_loader) // ACCUMULATION_STEPS) * EPOCHS
opt_scheduler = get_linear_schedule_with_warmup(
    opt_optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

print(f"Model: MLACIModel (multi-layer: YES)")
print(f"Adversarial λ: {LAMBDA_ADV} (OFF)")
print(f"Loss: CrossEntropy (no focal)")
print(f"Data: Augmented only (no oversampling)")
print(f"Parameters: {sum(p.numel() for p in opt_model.parameters()):,}")
print(f"Epochs: {EPOCHS}")

Model: MLACIModel (multi-layer: YES)
Adversarial λ: 0.0 (OFF)
Loss: CrossEntropy (no focal)
Data: Augmented only (no oversampling)
Parameters: 187,769,860
Epochs: 4


In [ ]:
# ============================================================================
# CELL E: Train — Aug + MultiLayer Only
# ============================================================================

print("\n" + "#"*80)
print("# EXPERIMENT: Aug + MultiLayer Only (no adv, no focal, no oversample)")
print("#"*80)

best_score = 0
best_results = {}
all_epoch_results = []

for epoch in range(EPOCHS):
    # --- TRAIN ---
    opt_model.train()
    total_loss = 0
    correct = 0
    total = 0
    opt_optimizer.zero_grad()

    for batch_idx, batch in enumerate(tqdm(opt_train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")):
        input_ids = batch['input_ids_orig'].to(device)
        attention_mask = batch['attention_mask_orig'].to(device)
        val_labels = batch['validity_label'].to(device)
        plaus_labels = batch['plausibility_label'].to(device)

        validity_logits, plausibility_logits = opt_model(input_ids, attention_mask)

        # Standard CE loss, no focal
        loss_val = opt_criterion(validity_logits, val_labels)
        # Adversarial loss computed but λ=0 so no effect
        loss_plaus = adv_criterion(plausibility_logits, plaus_labels)
        loss = (loss_val + LAMBDA_ADV * loss_plaus) / ACCUMULATION_STEPS

        loss.backward()

        if (batch_idx + 1) % ACCUMULATION_STEPS == 0 or (batch_idx + 1) == len(opt_train_loader):
            torch.nn.utils.clip_grad_norm_(opt_model.parameters(), 1.0)
            opt_optimizer.step()
            opt_scheduler.step()
            opt_optimizer.zero_grad()

        total_loss += loss.item() * ACCUMULATION_STEPS
        preds = torch.argmax(validity_logits, dim=1)
        correct += (preds == val_labels).sum().item()
        total += val_labels.size(0)

    train_acc = correct / total

    # --- EVALUATE ---
    opt_model.eval()
    all_preds = []
    all_val = []
    all_plaus = []
    idx = 0

    with torch.no_grad():
        for batch in opt_val_loader:
            input_ids = batch['input_ids_orig'].to(device)
            attention_mask = batch['attention_mask_orig'].to(device)
            val_labels = batch['validity_label'].to(device)
            bs = val_labels.size(0)

            validity_logits, _ = opt_model(input_ids, attention_mask)
            preds = torch.argmax(validity_logits, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_val.extend(val_labels.cpu().numpy())
            for i in range(bs):
                all_plaus.append(opt_val_plausibility[idx])
                idx += 1

    all_preds = np.array(all_preds)
    all_val = np.array(all_val)
    all_plaus = np.array(all_plaus)

    # Overall accuracy
    val_acc = accuracy_score(all_val, all_preds)

    # Per-condition accuracy
    conditions = {}
    for plaus in [0, 1]:
        for valid in [0, 1]:
            mask = (all_plaus == plaus) & (all_val == valid)
            if mask.sum() > 0:
                acc = accuracy_score(all_val[mask], all_preds[mask])
                plaus_name = "Plausible" if plaus == 1 else "Implausible"
                valid_name = "Valid" if valid == 1 else "Invalid"
                conditions[f"{plaus_name}_{valid_name}"] = acc

    # Content effect
    intra_plaus = abs(conditions.get('Plausible_Valid', 0) - conditions.get('Plausible_Invalid', 0))
    intra_implaus = abs(conditions.get('Implausible_Valid', 0) - conditions.get('Implausible_Invalid', 0))
    intra_effect = (intra_plaus + intra_implaus) / 2

    cross_valid = abs(conditions.get('Plausible_Valid', 0) - conditions.get('Implausible_Valid', 0))
    cross_invalid = abs(conditions.get('Plausible_Invalid', 0) - conditions.get('Implausible_Invalid', 0))
    cross_effect = (cross_valid + cross_invalid) / 2

    val_ce = (intra_effect + cross_effect) / 2

    # Combined score
    combined = val_acc * 100 / (1 + np.log(1 + val_ce * 100))

    print(f"\n  Epoch {epoch+1}: Train Acc={train_acc*100:.2f}% | Val Acc={val_acc*100:.2f}% | CE={val_ce*100:.2f}% | Score={combined:.2f}")
    print(f"    P-V={conditions.get('Plausible_Valid',0)*100:.1f}  P-I={conditions.get('Plausible_Invalid',0)*100:.1f}  "
          f"I-V={conditions.get('Implausible_Valid',0)*100:.1f}  I-I={conditions.get('Implausible_Invalid',0)*100:.1f}")

    epoch_result = {
        'epoch': epoch + 1,
        'train_acc': round(train_acc * 100, 2),
        'val_acc': round(val_acc * 100, 2),
        'val_ce': round(val_ce * 100, 2),
        'combined_score': round(combined, 2),
        'conditions': {k: round(v * 100, 2) for k, v in conditions.items()}
    }
    all_epoch_results.append(epoch_result)

    if combined > best_score:
        best_score = combined
        best_results = {
            'experiment': 'Aug + MultiLayer Only',
            'best_epoch': epoch + 1,
            'val_acc': round(val_acc * 100, 2),
            'val_ce': round(val_ce * 100, 2),
            'combined_score': round(combined, 2),
            'conditions': {k: round(v * 100, 2) for k, v in conditions.items()},
        }

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

best_results['all_epochs'] = all_epoch_results
print(f"\n{'='*60}")
print(f"★ BEST: Epoch {best_results['best_epoch']} | Acc={best_results['val_acc']}% | CE={best_results['val_ce']}% | Score={best_results['combined_score']}")
print(f"{'='*60}")


################################################################################
# EXPERIMENT: Aug + MultiLayer Only (no adv, no focal, no oversample)
################################################################################


Epoch 1/4: 100%|██████████| 510/510 [04:49<00:00,  1.76it/s]



  Epoch 1: Train Acc=68.28% | Val Acc=81.25% | CE=5.91% | Score=27.70
    P-V=80.6  P-I=74.3  I-V=86.1  I-I=83.8


Epoch 2/4: 100%|██████████| 510/510 [04:45<00:00,  1.79it/s]



  Epoch 2: Train Acc=88.58% | Val Acc=80.56% | CE=20.87% | Score=19.72
    P-V=55.6  P-I=91.4  I-V=77.8  I-I=97.3


Epoch 3/4: 100%|██████████| 510/510 [04:45<00:00,  1.79it/s]



  Epoch 3: Train Acc=95.88% | Val Acc=85.42% | CE=8.69% | Score=26.11
    P-V=88.9  P-I=74.3  I-V=91.7  I-I=86.5


Epoch 4/4: 100%|██████████| 510/510 [04:45<00:00,  1.79it/s]



  Epoch 4: Train Acc=98.19% | Val Acc=86.81% | CE=5.83% | Score=29.71
    P-V=86.1  P-I=80.0  I-V=91.7  I-I=89.2

★ BEST: Epoch 4 | Acc=86.81% | CE=5.83% | Score=29.71


In [ ]:
# ============================================================================
# CELL F: Save results & compare with all ablations
# ============================================================================

# Save standalone result
with open('optimal_experiment_results.json', 'w') as f:
    json.dump(best_results, f, indent=2)
print(" Saved to optimal_experiment_results.json")

# Load previous ablation results and add this one
try:
    with open('ablation_results.json', 'r') as f:
        all_results = json.load(f)
except FileNotFoundError:
    all_results = {}

all_results['optimal'] = best_results

with open('ablation_results_complete.json', 'w') as f:
    json.dump(all_results, f, indent=2)
print(" Saved complete results to ablation_results_complete.json")

# Print comparison table
print(f"\n{'='*80}")
print("COMPLETE COMPARISON (including new optimal experiment)")
print(f"{'='*80}")
print(f"{'Configuration':<35} {'Acc (%)':<10} {'CE (%)':<10} {'Score':<10}")
print("-"*65)

keys_order = ['full', 'no_aug', 'no_adv', 'no_focal', 'no_oversample', 'no_multilayer', 'optimal']
for key in keys_order:
    if key in all_results:
        r = all_results[key]
        marker = " ← NEW" if key == 'optimal' else ""
        print(f"{r['experiment']:<35} {r['val_acc']:<10} {r['val_ce']:<10} {r['combined_score']:<10}{marker}")

print("-"*65)
print(f"\nOfficial test submission:          79.06      4.17       29.92")

# Per-condition breakdown
print(f"\n{'='*80}")
print("PER-CONDITION ACCURACY (%)")
print(f"{'='*80}")
print(f"{'Config':<35} {'P-V':<8} {'P-I':<8} {'I-V':<8} {'I-I':<8}")
print("-"*67)
for key in keys_order:
    if key in all_results:
        r = all_results[key]
        c = r['conditions']
        print(f"{r['experiment']:<35} {c.get('Plausible_Valid',0):<8.1f} {c.get('Plausible_Invalid',0):<8.1f} "
              f"{c.get('Implausible_Valid',0):<8.1f} {c.get('Implausible_Invalid',0):<8.1f}")